# Read Generated `finfact-io` Data Directory

This notebook demonstrates how to call `finfact_io.data_directory.DataDirectoryReader` from a notebook that may not be started from the repository root.

It reads the generated datasets under `data/`, including CSI 300/500/2000 point-in-time index directories. It does not read or mutate the original raw source directories such as `/Users/sun/Downloads/指数数据` or `/Users/sun/Downloads/A股数据_每日指标`.


## 1. Configure Project Root

Set `FINFACT_PROJECT_ROOT` to the exact local path of the `finfact-io` repository. The notebook validates that `src/finfact_io/data_directory.py` exists under that path and raises an error immediately if it does not.

You can also set `FINFACT_GENERATED_DATA_DIR` when the generated `data/` directory is stored somewhere else.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

# User-specified repository root. Keep this explicit so the notebook never imports
# an unintended package when it is launched from another working directory.
FINFACT_PROJECT_ROOT = os.environ.get("FINFACT_PROJECT_ROOT", "/Users/sun/Documents/Code/finfact-io")
PROJECT_ROOT = Path(FINFACT_PROJECT_ROOT).expanduser().resolve()

DATA_DIRECTORY_MODULE = PROJECT_ROOT / "src" / "finfact_io" / "data_directory.py"
if not DATA_DIRECTORY_MODULE.is_file():
    raise FileNotFoundError(
        "Cannot find src/finfact_io/data_directory.py under FINFACT_PROJECT_ROOT. "
        f"FINFACT_PROJECT_ROOT={PROJECT_ROOT!s}"
    )

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from finfact_io.data_directory import DataDirectoryReader

DATA_DIR = Path(os.environ.get("FINFACT_GENERATED_DATA_DIR", PROJECT_ROOT / "data")).expanduser().resolve()
reader = DataDirectoryReader(DATA_DIR)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIRECTORY_MODULE:", DATA_DIRECTORY_MODULE)
print("DATA_DIR:", DATA_DIR)

: 

## 2. Inspect Available Generated Datasets

`DataDirectoryReader` expects each generated dataset to contain its own manifest and quality report. This quick inspection confirms the expected directory structure before loading analytical tables.

In [ ]:
DATASETS = [
    "ashare_daily_metrics",
    "csi300",
    "csi500",
    "csi2000",
    "industry_sw_current_reference",
    "listing_board_current_reference",
]

for dataset_name in DATASETS:
    dataset_path = reader.dataset_path(dataset_name)
    manifest = reader.manifest(dataset_name)
    quality = reader.quality(dataset_name)
    status_counts = quality["status"].value_counts(dropna=False).to_dict() if "status" in quality.columns else {}

    print(f"{dataset_name}")
    print(f"  path: {dataset_path}")
    print(f"  manifest dataset: {manifest.get('dataset')}")
    print(f"  quality rows: {len(quality)}, status counts: {status_counts}")


## 3. Read A-share Daily Metrics by Date

The A-share daily metrics dataset is organized by trading day. Columns are standardized in English, while financial units are preserved in the schema file.

In [ ]:
ashare_schema = reader.ashare_daily_schema()
ashare_dates = reader.ashare_daily_dates()

print("date range:", ashare_dates["date"].iloc[0], "to", ashare_dates["date"].iloc[-1])
print("number of trading-day files:", len(ashare_dates))
display(ashare_schema[["standard_column", "raw_column", "unit"]])

sample_daily = reader.ashare_daily_by_date(
    "2026-06-03",
    symbols=["000001.SZ", "000002.SZ"],
)

display(sample_daily[["symbol", "trade_date", "open", "close", "amount", "total_market_cap"]])

## 4. Read CSI 300, CSI 500, and CSI 2000 Index Data

The generated CSI point-in-time index datasets share the generic `reader.index_*` API. `csi300` is the repository dataset for 沪深300 (`000300.SH`).

This example reads a small three-trading-day price window and one day of as-of constituent weights, so it stays lightweight even for CSI 2000.


In [ ]:
INDEX_DATASETS = {
    "csi300": {"name": "沪深300", "expected_members": 300},
    "csi500": {"name": "中证500", "expected_members": 500},
    "csi2000": {"name": "中证2000", "expected_members": 2000},
}
SAMPLE_INDEX_DATE = "2026-06-03"
SAMPLE_INDEX_START = "2026-06-01"

index_summaries = []
index_daily_samples = {}
index_weight_samples = {}
index_top_weight_samples = []

for dataset_name, config in INDEX_DATASETS.items():
    manifest = reader.manifest(dataset_name)
    quality = reader.quality(dataset_name)
    snapshot_index = reader.index_weight_snapshots(dataset_name)
    daily_asof_index = reader.index_daily_asof_dates(dataset_name)
    daily = reader.index_daily(dataset_name, start=SAMPLE_INDEX_START, end=SAMPLE_INDEX_DATE)
    weights = reader.index_weights_by_date(dataset_name, SAMPLE_INDEX_DATE)

    index_daily_samples[dataset_name] = daily.assign(dataset=dataset_name)
    index_weight_samples[dataset_name] = weights
    index_top_weight_samples.append(
        weights.sort_values("weight", ascending=False)
        .head(5)
        .assign(dataset=dataset_name)
    )

    duplicate_member_rows = int(weights.duplicated(["date", "member_symbol"]).sum())
    index_summaries.append(
        {
            "dataset": dataset_name,
            "index_code": manifest["index_code"],
            "index_name": manifest["index_name"],
            "expected_members": config["expected_members"],
            "daily_rows": len(daily),
            "snapshot_dates": len(snapshot_index),
            "asof_trading_days": len(daily_asof_index),
            "sample_date": SAMPLE_INDEX_DATE,
            "member_rows": len(weights),
            "unique_members": weights["member_symbol"].nunique(),
            "duplicate_member_rows": duplicate_member_rows,
            "weight_sum": float(weights["weight"].sum()),
            "latest_weight_snapshot": weights["weight_snapshot_date"].iloc[0],
            "total_return_available": bool(manifest.get("total_return", {}).get("available", False)),
            "quality_status_counts": quality["status"].value_counts(dropna=False).to_dict(),
        }
    )

index_summary = pd.DataFrame(index_summaries)
index_daily_sample = pd.concat(index_daily_samples.values(), ignore_index=True)
index_top_weights = pd.concat(index_top_weight_samples, ignore_index=True)

display(index_summary)
display(
    index_daily_sample[
        [
            "dataset",
            "date",
            "index_code",
            "index_name",
            "close",
            "return_pct",
            "total_return",
            "total_return_available",
        ]
    ]
)
display(
    index_top_weights[
        [
            "dataset",
            "date",
            "index_code",
            "member_symbol",
            "weight",
            "weight_snapshot_date",
            "effective_date",
            "quality_status",
        ]
    ]
)


## 5. Join Stock-level Dimensions

Industry and listing-board references are independent stock dimensions. Missing symbols are retained, which makes unmatched records easy to audit.

In [ ]:
sample_symbols = ["000001.SZ", "688001.SH", "999999.SZ"]

stock_dimensions = reader.join_stock_dimensions(
    symbols=sample_symbols,
    include_industry=True,
    include_listing_board=True,
)

display(stock_dimensions)

## 6. Build Industry and Listing-board Exposure Matrices

Both matrices use stocks as rows and categories as columns. Active exposure follows the common matrix expression `G.T @ (w - b)`.

In [ ]:
industry_matrix = reader.sw_industry_matrix(level="L1", stocks=sample_symbols)
board_matrix = reader.listing_board_matrix(stocks=sample_symbols)

portfolio_weights = pd.Series({"000001.SZ": 0.60, "688001.SH": 0.40}, name="portfolio_weight")
benchmark_weights = pd.Series({"000001.SZ": 0.50, "688001.SH": 0.20, "999999.SZ": 0.30}, name="benchmark_weight")

active_weight = portfolio_weights.sub(benchmark_weights, fill_value=0.0).reindex(sample_symbols).fillna(0.0)
industry_active_exposure = industry_matrix.T @ active_weight
board_active_exposure = board_matrix.T @ active_weight

print("Industry matrix shape:", industry_matrix.shape)
display(industry_matrix)
display(industry_active_exposure.rename("active_exposure").loc[lambda s: s != 0])

print("Listing-board matrix shape:", board_matrix.shape)
display(board_matrix)
display(board_active_exposure.rename("active_exposure").loc[lambda s: s != 0])

## 7. Run Basic Data Quality Checks

These checks verify the current generated listing-board reference and the CSI 300/500/2000 point-in-time constituent weights used above. CSI 2000 may report a `total_return_unavailable` warning when the local generated directory cannot find its total-return index member; fail/error quality statuses are still treated as blocking.


In [ ]:
listed_board_counts = reader.listing_board_counts(scope="listed").sort_values("listing_board_code")
observed_listed_counts = dict(zip(listed_board_counts["listing_board_code"], listed_board_counts["stock_count"]))
expected_listed_counts = {
    "BSE": 317,
    "CHINEXT": 1398,
    "MAIN": 3199,
    "STAR": 610,
}

display(listed_board_counts)
assert observed_listed_counts == expected_listed_counts

index_quality_checks = []
for dataset_name, config in INDEX_DATASETS.items():
    weights = index_weight_samples[dataset_name]
    quality = reader.quality(dataset_name)
    quality_status = quality["status"].astype(str).str.lower()

    blocking_quality = quality[quality_status.isin(["fail", "error"])]
    if not blocking_quality.empty:
        display(blocking_quality)
        raise AssertionError(f"{dataset_name} has blocking quality statuses")

    warning_checks = quality.loc[quality_status.eq("warning"), "check"].astype(str).tolist()
    if dataset_name == "csi2000":
        unexpected_warning_checks = sorted(set(warning_checks) - {"total_return_unavailable"})
    else:
        unexpected_warning_checks = warning_checks
    assert not unexpected_warning_checks, f"{dataset_name} unexpected warnings: {unexpected_warning_checks}"

    member_rows = len(weights)
    unique_members = weights["member_symbol"].nunique()
    duplicate_member_rows = int(weights.duplicated(["date", "member_symbol"]).sum())
    weight_sum = float(weights["weight"].sum())

    assert member_rows == config["expected_members"], dataset_name
    assert unique_members == config["expected_members"], dataset_name
    assert duplicate_member_rows == 0, dataset_name
    assert abs(weight_sum - 100.0) < 0.05, dataset_name

    index_quality_checks.append(
        {
            "dataset": dataset_name,
            "expected_members": config["expected_members"],
            "member_rows": member_rows,
            "unique_members": unique_members,
            "duplicate_member_rows": duplicate_member_rows,
            "weight_sum": weight_sum,
            "warning_checks": warning_checks,
        }
    )

index_quality_checks = pd.DataFrame(index_quality_checks)
display(index_quality_checks)


## Troubleshooting

- If import fails, check that `FINFACT_PROJECT_ROOT` points to the repository root containing `src/finfact_io/data_directory.py`.
- If data loading fails, check that `FINFACT_GENERATED_DATA_DIR` points to the generated data directory, or that `data/` exists under `FINFACT_PROJECT_ROOT`.
- If a quality assertion fails after regenerating data, inspect the relevant dataset's `data_quality.csv` before using it for analysis.
